<a href="https://colab.research.google.com/github/xiaoyu777/xiaoyu777.github.io/blob/main/%E7%A0%B4%E4%BC%A4%E9%A3%8EICU%E5%9F%B9%E8%AE%AD%E8%AF%BE%E4%BB%B6%E7%94%9F%E6%88%90%E5%99%A8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pptx
from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.dml.color import RGBColor
from pptx.enum.text import PP_ALIGN, MSO_ANCHOR
from pptx.enum.shapes import MSO_SHAPE
from datetime import datetime

# ==========================================
# 配置参数与配色方案
# ==========================================
FILENAME = f"破伤风ICU管理_住院医师培训_{datetime.now().strftime('%Y%m%d')}.pptx"

# 配色 (深色系医疗风格)
COLOR_BG = RGBColor(12, 35, 64)       # 深海蓝 (背景)
COLOR_TITLE = RGBColor(255, 255, 255) # 纯白 (标题)
COLOR_TEXT_MAIN = RGBColor(220, 230, 241) # 浅灰蓝 (正文)
COLOR_ACCENT = RGBColor(255, 140, 0)  # 活力橙 (强调/关键点)
COLOR_SUBTLE = RGBColor(128, 128, 128)# 灰色 (引用)

# 字体大小配置
SIZE_TITLE = Pt(36)
SIZE_BODY = Pt(16)
SIZE_NOTE = Pt(11)

def create_presentation():
    prs = Presentation()

    # 定义母版背景为深色
    # 注意：python-pptx直接修改母版背景较复杂，这里采用通过形状填充模拟背景的方法
    # 在实际应用中，建议使用预设的深色模板，这里为了代码独立性采用纯代码生成

    slides_content = get_slides_data()

    for slide_data in slides_content:
        layout_index = slide_data.get('layout', 1) # 默认使用标题+内容版式
        slide = prs.slides.add_slide(prs.slide_layouts[layout_index])

        # 1. 设置背景颜色 (通过覆盖一个矩形实现，确保兼容性)
        background = slide.shapes.add_shape(
            MSO_SHAPE.RECTANGLE, 0, 0, prs.slide_width, prs.slide_height
        )
        background.fill.solid()
        background.fill.fore_color.rgb = COLOR_BG
        background.line.fill.background() # 去除边框
        # 将背景置于底层 (简单处理：后续添加的元素自然在顶层)

        # 2. 添加标题
        if 'title' in slide_data:
            title_shape = slide.shapes.title
            if title_shape:
                title_shape.text = slide_data['title']
                title_shape.text_frame.paragraphs[0].font.color.rgb = COLOR_TITLE
                title_shape.text_frame.paragraphs[0].font.size = SIZE_TITLE
                title_shape.text_frame.paragraphs[0].font.bold = True
                # 确保标题可见（移到顶层）
                slide.shapes._spTree.insert(2, title_shape._element)

        # 3. 添加正文内容
        if 'content' in slide_data:
            # 自动调整文本框位置，避开标题
            left = Inches(0.5)
            top = Inches(1.5)
            width = Inches(9)
            height = Inches(5.5)

            textbox = slide.shapes.add_textbox(left, top, width, height)
            text_frame = textbox.text_frame
            text_frame.word_wrap = True

            for line in slide_data['content']:
                p = text_frame.add_paragraph()
                p.text = line['text']
                p.font.size = SIZE_BODY
                p.font.color.rgb = COLOR_ACCENT if line.get('highlight') else COLOR_TEXT_MAIN
                p.level = line.get('level', 0)
                p.space_after = Pt(10)

                if line.get('bold'):
                    p.font.bold = True

        # 4. 特殊元素处理 (表格)
        if 'table' in slide_data:
            create_table(slide, slide_data['table'])

        # 5. 添加视觉元素占位符 (如果指定)
        if 'visual' in slide_data:
            create_visual_placeholder(slide, slide_data['visual'])

        # 6. 添加引用/脚注
        if 'ref' in slide_data:
            add_reference(slide, slide_data['ref'])

        # 7. 添加演讲者备注 (Speaker Notes)
        if 'notes' in slide_data:
            notes_slide = slide.notes_slide
            text_frame = notes_slide.notes_text_frame
            text_frame.text = slide_data['notes']

    prs.save(FILENAME)
    print(f"成功生成PPT: {FILENAME}")

def create_table(slide, table_data):
    rows = len(table_data['data']) + 1
    cols = len(table_data['headers'])

    left = Inches(1)
    top = Inches(2.0)
    width = Inches(8)
    height = Inches(0.8 * rows)

    shape = slide.shapes.add_table(rows, cols, left, top, width, height)
    table = shape.table

    # 设置表头
    for i, header in enumerate(table_data['headers']):
        cell = table.cell(0, i)
        cell.text = header
        cell.fill.solid()
        cell.fill.fore_color.rgb = RGBColor(40, 60, 90) # 深一点的蓝色
        paragraph = cell.text_frame.paragraphs[0]
        paragraph.font.color.rgb = COLOR_ACCENT
        paragraph.font.bold = True
        paragraph.font.size = Pt(14)
        paragraph.alignment = PP_ALIGN.CENTER

    # 填充数据
    for row_idx, row_data in enumerate(table_data['data']):
        for col_idx, item in enumerate(row_data):
            cell = table.cell(row_idx + 1, col_idx)
            cell.text = str(item)
            cell.fill.solid()
            cell.fill.fore_color.rgb = RGBColor(20, 45, 75) if row_idx % 2 == 0 else RGBColor(25, 50, 80)
            paragraph = cell.text_frame.paragraphs[0]
            paragraph.font.color.rgb = COLOR_TEXT_MAIN
            paragraph.font.size = Pt(13)
            paragraph.alignment = PP_ALIGN.LEFT

def create_visual_placeholder(slide, text):
    # 创建一个带有边框的矩形，模拟图表位置
    left = Inches(6)
    top = Inches(2)
    width = Inches(3.5)
    height = Inches(3.5)

    shape = slide.shapes.add_shape(MSO_SHAPE.ROUNDED_RECTANGLE, left, top, width, height)
    shape.fill.solid()
    shape.fill.fore_color.rgb = RGBColor(30, 50, 80)
    shape.line.color.rgb = COLOR_ACCENT
    shape.line.width = Pt(2)

    tf = shape.text_frame
    p = tf.paragraphs[0]
    p.text = f"[视觉元素]\n{text}"
    p.font.color.rgb = COLOR_TEXT_MAIN
    p.alignment = PP_ALIGN.CENTER

def add_reference(slide, text):
    left = Inches(0.5)
    top = Inches(7.0)
    width = Inches(9)
    height = Inches(0.4)

    textbox = slide.shapes.add_textbox(left, top, width, height)
    p = textbox.text_frame.paragraphs[0]
    p.text = f"Ref: {text}"
    p.font.size = SIZE_NOTE
    p.font.color.rgb = COLOR_SUBTLE

def get_slides_data():
    return [
        # --- Slide 1: 封面 ---
        {
            'layout': 0, # Title Slide
            'title': "破伤风感染：ICU重症管理全流程",
            'content': [
                {'text': "Tetanus: From Diagnosis to ICU Management", 'highlight': False},
                {'text': "\n汇报人：ICU教学组", 'highlight': False},
                {'text': "对象：ICU住院医师 (PGY 1-3)", 'highlight': False},
                {'text': datetime.now().strftime("%Y年%m月"), 'highlight': False}
            ],
            'notes': "开场语：大家好。破伤风虽然是古老的疾病，但在ICU中其死亡率依然高达10%-20%，主要死因已从早期的呼吸衰竭转变为自主神经功能紊乱和院内感染。今天我们将基于2020-2025年的最新证据，梳理重症管理策略。"
        },

        # --- Slide 2: 病例引入 ---
        {
            'title': "病例引入：并非所有伤口都显而易见",
            'content': [
                {'text': "患者：56岁，男性，建筑工人", 'bold': True},
                {'text': "主诉：张口困难2天，颈部僵硬1天", 'level': 0},
                {'text': "现病史：", 'highlight': True, 'level': 0},
                {'text': "10天前右足底被锈钉刺伤，自行碘伏处理，未接种疫苗。", 'level': 1},
                {'text': "入院后出现阵发性腹肌紧张，压舌板试验阳性。", 'level': 1},
                {'text': "思考：", 'highlight': True, 'level': 0},
                {'text': "1. 该患者Ablett分级是多少？", 'level': 1},
                {'text': "2. 伤口已结痂，还需要清创吗？", 'level': 1},
                {'text': "3. 入ICU的第一要务是什么？", 'level': 1}
            ],
            'visual': "苦笑面容 (Risus Sardonicus) \n或 角弓反张 示意图",
            'notes': "教学点：1. 潜伏期3-21天，潜伏期<7天预示重症。2. 强调'不典型伤口'，约20%患者无明显伤口史。3. 压舌板试验是简便的床旁诊断工具。这个病例是典型的中重度破伤风起病。"
        },

        # --- Slide 3: 流行病学 ---
        {
            'title': "全球与中国流行病学概况",
            'content': [
                {'text': "全球现状：", 'bold': True},
                {'text': "主要见于低收入国家，但高收入国家老年人（免疫衰退）发病率回升。", 'level': 1},
                {'text': "中国数据：", 'bold': True},
                {'text': "非新生儿破伤风呈散发状态，多见于无免疫史的成人。", 'level': 1},
                {'text': "死亡率：ICU环境下约为 10-20% (重症组)。", 'highlight': True, 'level': 1},
                {'text': "高危人群：>60岁，农业/建筑工人，糖尿病足患者。", 'level': 1}
            ],
            'ref': "Yen LM, Thwaites CL. Lancet. 2019; Kyu HH, et al. Lancet Infect Dis. 2020.",
            'notes': "注意：中国目前是非新生儿破伤风为主。重点关注老年人群，因为他们的疫苗保护力已随时间衰退。这是我们ICU收治的主要群体。"
        },

        # --- Slide 4: 病原与发病机制 ---
        {
            'title': "核心机制：破伤风痉挛毒素 (TeNT)",
            'content': [
                {'text': "破伤风梭菌 (Clostridium tetani)：", 'bold': True},
                {'text': "革兰阳性厌氧杆菌，芽孢呈鼓槌状，广泛存在于土壤。", 'level': 1},
                {'text': "毒素作用机制 (双重打击)：", 'highlight': True},
                {'text': "1. 逆行轴浆运输：沿运动神经元上行至脊髓/脑干。", 'level': 1},
                {'text': "2. 突触阻断：裂解VAMP-2蛋白，阻断抑制性递质(GABA, 甘氨酸)释放。", 'level': 1},
                {'text': "结果：α运动神经元去抑制 -> 肌肉强直/痉挛；交感神经去抑制 -> 自主神经风暴。", 'bold': True, 'level': 1}
            ],
            'visual': "机制图解：\n毒素阻断突触前膜\nGABA释放过程",
            'notes': "一定要理解：抗毒素只能中和血循环中的游离毒素，对已结合神经突触的毒素无效。这就是为什么破伤风病程长（需神经末梢再生，约4-6周）。"
        },

        # --- Slide 5: 临床表现与诊断 ---
        {
            'title': "临床诊断：不仅仅是张口受限",
            'content': [
                {'text': "典型四联征：", 'bold': True},
                {'text': "张口受限 (Trismus) -> 苦笑面容 -> 颈项强直 -> 角弓反张", 'level': 1},
                {'text': "诊断依据：完全依赖临床！", 'highlight': True},
                {'text': "不需要细菌培养（阳性率低，仅30%）。", 'level': 1},
                {'text': "血清抗体水平通常 <0.01 IU/mL (无保护性)。", 'level': 1},
                {'text': "鉴别诊断：", 'bold': True},
                {'text': "斯特林毒药中毒 (无牙关紧闭)、牙源性感染、低钙血症、抗精神病药物反应。", 'level': 1}
            ],
            'notes': "ICU医生要警惕不典型的局灶性破伤风，可能进展为全身型。任何不明原因的肌肉痉挛都要排查外伤史。"
        },

        # --- Slide 6: Ablett 分级 (关键决策工具) ---
        {
            'title': "严重程度评估：Ablett 分级",
            'table': {
                'headers': ['分级', '临床特征', 'ICU干预建议'],
                'data': [
                    ['I (轻度)', '轻度张口受限，无痉挛，无吞咽困难', '普通病房，观察'],
                    ['II (中度)', '明显张口受限，强直，轻度痉挛，无呼吸窘迫', 'ICU监护，准备气道'],
                    ['III (重度)', '严重痉挛，吞咽困难，呼吸窘迫', 'ICU，气管切开，机械通气'],
                    ['IV (极重度)', 'III级症状 + 自主神经功能紊乱 (风暴)', 'ICU，深镇静，血流动力学控制']
                ]
            },
            'notes': "Ablett分级决定了资源分配。III级和IV级必须进ICU。IV级的核心特征是自主神经功能不稳定（血压忽高忽低，心率快慢交替），这是目前致死的主要原因。",
            'ref': "Ablett JJ. Tetanus. 1967."
        },

        # --- Slide 7: ICU管理总览 ---
        {
            'title': "ICU管理策略：核心三支柱",
            'content': [
                {'text': "1. 清除毒素来源 (Source Control)", 'bold': True},
                {'text': "抗生素 + 伤口清创", 'level': 1},
                {'text': "2. 中和游离毒素 (Neutralization)", 'bold': True},
                {'text': "HTIG (人破伤风免疫球蛋白)", 'level': 1},
                {'text': "3. 症状控制与支持 (Symptomatic Control)", 'highlight': True, 'bold': True},
                {'text': "痉挛控制 (镇静/肌松)", 'level': 1},
                {'text': "自主神经管理", 'level': 1},
                {'text': "气道与呼吸支持", 'level': 1}
            ],
            'visual': "流程图：\n入院 -> 镇静/抗毒素 ->\n清创 -> ICU综合支持",
            'notes': "这三根支柱缺一不可。注意顺序：先用抗毒素，再清创！避免清创导致大量毒素入血。"
        },

        # --- Slide 8: 抗毒素与抗生素 ---
        {
            'title': "步骤一：中和毒素与抗感染",
            'content': [
                {'text': "抗毒素治疗 (越早越好)", 'bold': True},
                {'text': "首选：HTIG 3000-6000 IU 肌注 (部分用于伤口周围)。", 'highlight': True, 'level': 1},
                {'text': "替代：马破伤风抗毒素 (TAT) 1500-3000 IU (过敏风险高，需皮试)。", 'level': 1},
                {'text': "最新观点：WHO推荐500 IU HTIG可能已足够，但重症常沿用大剂量。", 'level': 1},
                {'text': "抗生素选择", 'bold': True},
                {'text': "首选：甲硝唑 500mg IV q6-8h (7-10天)。", 'highlight': True, 'level': 1},
                {'text': "替代：青霉素G。", 'level': 1},
                {'text': "为何首选甲硝唑？青霉素是GABA受体拮抗剂，理论上可能加重痉挛。", 'level': 1}
            ],
            'ref': "WHO Position Paper 2017; Rodrigo C, et al. Cochrane Database Syst Rev. 2012.",
            'notes': "关于HTIG剂量有争议，但对于ICU重症患者，通常给予3000 IU。鞘内注射目前仍属Label off使用，虽然有文献支持可缩短病程。"
        },

        # --- Slide 9: 痉挛控制 (镇静策略) ---
        {
            'title': "步骤二：肌痉挛控制策略",
            'content': [
                {'text': "环境控制：", 'bold': True},
                {'text': "单间、避光、隔音 (减少声光刺激诱发痉挛)。", 'level': 1},
                {'text': "药物阶梯治疗：", 'highlight': True},
                {'text': "一线：苯二氮卓类 (Benzodiazepines)", 'bold': True, 'level': 1},
                {'text': "咪达唑仑 (5-15mg/h) 或 地西泮。需极高剂量，注意耐受性。", 'level': 2},
                {'text': "二线：非去极化肌松药 (NMBAs)", 'bold': True, 'level': 1},
                {'text': "维库溴铵/罗库溴铵。当镇静无法控制痉挛导致通气受损时使用。", 'level': 2},
                {'text': "辅助：丙泊酚 (注意PRIS风险)", 'level': 1}
            ],
            'notes': "不用害怕大剂量镇静。破伤风患者对镇静剂的耐受性极高。目标是控制强直发作，同时避免过度抑制呼吸（如果已插管则无需顾虑呼吸抑制）。"
        },

        # --- Slide 10: 自主神经风暴管理 (难点) ---
        {
            'title': "难点攻关：自主神经风暴",
            'content': [
                {'text': "定义：Ablett IV级特征，表现为高热、血压剧烈波动、心动过速、多汗。", 'level': 0},
                {'text': "核心药物：硫酸镁 (Magnesium Sulfate)", 'bold': True, 'highlight': True},
                {'text': "机制：突触前抑制儿茶酚胺释放，钙拮抗作用。", 'level': 1},
                {'text': "给药：负荷 40mg/kg -> 维持 2g/h。", 'level': 1},
                {'text': "目标：血镁维持在 2.0 - 4.0 mmol/L。", 'level': 1},
                {'text': "监测：膝腱反射消失是中毒先兆，警惕低血压/心动过缓。", 'level': 1},
                {'text': "二线药物：", 'bold': True},
                {'text': "拉贝洛尔 (α+β阻滞)、右美托咪定 (α2激动剂，兼具镇静与交感抑制)。", 'level': 1}
            ],
            'visual': "Mg++\n监测指标\n呼吸/尿量/反射",
            'ref': "Thwaites CL. Cochrane Database Syst Rev. 2013 (Magnesium for tetanus).",
            'notes': "硫酸镁是高性价比的药物，不需额外通气支持即可使用。但在ICU中，联合使用右美托咪定（Dexmedetomidine）正成为新趋势，因为它可以减少苯二氮卓类的用量。"
        },

        # --- Slide 11: 气道管理与机械通气 ---
        {
            'title': "气道管理与机械通气",
            'content': [
                {'text': "早期气管切开 (Early Tracheostomy)", 'bold': True, 'highlight': True},
                {'text': "指征：Ablett III/IV级，预计通气时间 >10天。", 'level': 1},
                {'text': "时机：建议在插管后24小时内评估，尽早实施。", 'level': 1},
                {'text': "获益：防止喉头痉挛致死，便于气道清理，减少镇静需求。", 'level': 1},
                {'text': "机械通气策略：", 'bold': True},
                {'text': "无特异性模式，保护性肺通气原则。", 'level': 1},
                {'text': "注意：痉挛发作时会出现高气道压报警，此时应加深镇静/肌松，而非单纯调高压力上限。", 'level': 1}
            ],
            'notes': "为什么强调早气切？因为喉头痉挛是猝死主因。插管刺激可能诱发痉挛，气切耐受性更好。不要等到插管并发症出现再气切。"
        },

        # --- Slide 12: 营养与并发症 ---
        {
            'title': "营养支持与并发症预防",
            'content': [
                {'text': "高代谢状态：", 'bold': True},
                {'text': "痉挛消耗巨大能量，需高热量高蛋白支持。", 'level': 1},
                {'text': "途径：首选肠内营养 (鼻空肠管)，防止反流误吸。", 'level': 1},
                {'text': "常见并发症：", 'bold': True},
                {'text': "院内感染 (VAP, 导管相关感染)", 'level': 1},
                {'text': "深静脉血栓 (DVT) - 需预防性抗凝", 'level': 1},
                {'text': "椎体骨折 (剧烈痉挛所致)", 'level': 1},
                {'text': "消化道出血 (应激性溃疡)", 'level': 1}
            ],
            'notes': "鼻饲管要在镇静深度足够时放置，否则操作本身会诱发严重喉痉挛。"
        },

        # --- Slide 13: 伤口处理与预防 ---
        {
            'title': "伤口处理与免疫接种",
            'content': [
                {'text': "清创时机：", 'highlight': True},
                {'text': "必须在注射抗毒素 (HTIG) 后 1-6小时进行！", 'bold': True, 'level': 0},
                {'text': "彻底清除坏死组织和异物，破坏厌氧环境。", 'level': 1},
                {'text': "主动免疫 (疫苗接种)：关键知识点", 'highlight': True},
                {'text': "破伤风感染后不会产生自然免疫力 (致死毒素量 < 免疫原性量)。", 'level': 1},
                {'text': "出院前必须制定 Tt/Tdap 疫苗接种计划。", 'level': 1},
                {'text': "未免疫者：即刻、4周、6个月各接种一剂。", 'level': 1}
            ],
            'notes': "这是一个常考点，也是临床易错点：患者得过破伤风不代表有抗体，必须重新接种疫苗。"
        },

        # --- Slide 14: 新进展与预后 ---
        {
            'title': "预后评估与新进展",
            'content': [
                {'text': "预后评分：Tetanus Severity Score (TSS)", 'level': 0},
                {'text': "包含年龄、潜伏期、入院时间等指标，>8分死亡率极高。", 'level': 1},
                {'text': "新疗法探索：", 'bold': True},
                {'text': "1. 鞘内注射 HTIG：", 'level': 1},
                {'text': "理论：跨过血脑屏障直接中和毒素。Meta分析显示可缩短住院时间，但需更多RCT支持。", 'level': 2},
                {'text': "2. 维生素 C：", 'level': 1},
                {'text': "作为抗氧化剂辅助治疗，部分研究显示可降低死亡率 (Level 3 Evidence)。", 'level': 2}
            ],
            'ref': "Wiersinga WJ. JAMA. 2023; Hemmes SN. Intensive Care Med. 2023.",
            'notes': "鞘内注射是一个热门研究方向，如果遇到极重度患者，在常规治疗无效时，经过伦理批准和家属沟通，可作为挽救性治疗尝试。"
        },

        # --- Slide 15: 总结与思考 ---
        {
            'title': "Take-Home Messages",
            'content': [
                {'text': "诊断：依靠临床表现，Spatula Test (压舌板试验) 高度敏感。", 'level': 0},
                {'text': "决策：使用 Ablett 分级筛选 ICU 患者。", 'level': 0},
                {'text': "治疗核心：抗毒素 + 抗生素 + 镇静 + 气道保护。", 'highlight': True, 'bold': True, 'level': 0},
                {'text': "ICU重点：硫酸镁是控制自主神经风暴的基石；尽早气切。", 'level': 0},
                {'text': "预防：感染后免疫力不持久，必须补种疫苗。", 'level': 0}
            ],
            'notes': "请记住：破伤风是可以预防的疾病，但在ICU中它是极具挑战的重症。细致的护理和对并发症的预判是存活的关键。"
        },

        # --- Slide 16: 临床思考题 ---
        {
            'title': "临床思考题 (Discussion)",
            'content': [
                {'text': "Q1: 患者入院时伤口已愈合，是否还需要切开探查？", 'bold': True},
                {'text': "提示：厌氧环境可能存在于痂下。", 'level': 1},
                {'text': "Q2: 使用大剂量镇静剂时，如何评估患者的神志和病情变化？", 'bold': True},
                {'text': "提示：每日唤醒的可行性 vs 痉挛风险。", 'level': 1},
                {'text': "Q3: 为什么青霉素不再是首选抗生素？", 'bold': True},
                {'text': "提示：GABA拮抗作用。", 'level': 1}
            ],
            'notes': "引导住院医进行讨论。Q1答案是肯定的，需要探查。Q2是一个ICU难题，通常在破伤风急性期不建议每日唤醒，推荐使用脑电监测。"
        },

        # --- Slide 17: 参考文献 ---
        {
            'title': "参考文献 (References)",
            'content': [
                {'text': "[1] Yen LM, Thwaites CL. Tetanus. Lancet. 2019;393(10181):1659-1668. doi:10.1016/S0140-6736(18)33131-3", 'level': 0},
                {'text': "[2] WHO. Tetanus vaccines: WHO position paper – February 2017. Wkly Epidemiol Rec. 2017;92(6):53-76.", 'level': 0},
                {'text': "[3] Rodrigo C, et al. Pharmacological management of tetanus: an evidence-based review. Crit Care. 2014;18(2):217.", 'level': 0},
                {'text': "[4] Ha H, et al. Utility of Magnesium Sulfate in the Treatment of Tetanus. J Intensive Care Med. 2020.", 'level': 0},
                {'text': "[5] 中国破伤风免疫预防专家共识. 中华外科杂志, 2018.", 'level': 0}
            ],
            'notes': "建议阅读 Lancet 2019年的综述，是目前最全面的文献。"
        }
    ]

if __name__ == "__main__":
    create_presentation()

成功生成PPT: 破伤风ICU管理_住院医师培训_20260209.pptx


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [2]:
!pip install python-pptx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.8/472.8 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 13.3 MB/s eta 0:00:00


In [ ]:
# @title AI prompt cell

import ipywidgets as widgets
from IPython.display import display, HTML, Markdown,clear_output
from google.colab import ai

dropdown = widgets.Dropdown(
    options=[],
    layout={'width': 'auto'}
)

def update_model_list(new_options):
    dropdown.options = new_options
update_model_list(ai.list_models())

text_input = widgets.Textarea(
    placeholder='Ask me anything....',
    layout={'width': 'auto', 'height': '100px'},
)

button = widgets.Button(
    description='Submit Text',
    disabled=False,
    tooltip='Click to submit the text',
    icon='check'
)

output_area = widgets.Output(
     layout={'width': 'auto', 'max_height': '300px','overflow_y': 'scroll'}
)

def on_button_clicked(b):
    with output_area:
        output_area.clear_output(wait=False)
        accumulated_content = ""
        for new_chunk in ai.generate_text(prompt=text_input.value, model_name=dropdown.value, stream=True):
            if new_chunk is None:
                continue
            accumulated_content += new_chunk
            clear_output(wait=True)
            display(Markdown(accumulated_content))

button.on_click(on_button_clicked)
vbox = widgets.GridBox([dropdown, text_input, button, output_area])

display(HTML("""
<style>
.widget-dropdown select {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
.widget-textarea textarea {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
</style>
"""))
display(vbox)
